In [14]:
import requests
import json
import time

headers = {
  'authority': 'api.sofascore.com',
  'accept': '*/*',
  'accept-language': 'en-GB,en-US;q=0.9,en;q=0.8,es;q=0.7',
  'cache-control': 'max-age=0',
  'origin': 'https://www.sofascore.com',
  'referer': 'https://www.sofascore.com/',
  'sec-ch-ua': '"Chromium";v="122", "Not(A:Brand";v="24", "Google Chrome";v="122"',
  'sec-ch-ua-mobile': '?0',
  'sec-ch-ua-platform': '"macOS"',
  'sec-fetch-dest': 'empty',
  'sec-fetch-mode': 'cors',
  'sec-fetch-site': 'same-site',
  'user-agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_12_6) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/103.0.0.0 Safari/537.3'
  }

Elegir tipo de data

In [15]:
# data = 'statistics'
data = 'shotmap'
# data = 'lineups'

## Scrape

Let's scrape matches id from rounds

In [47]:
import requests
url = f'https://api.sofascore.com/api/v1/unique-tournament/240/season/58043/rounds'
response = requests.get(url, headers=headers)
current_mw = response.json()['currentRound']['round']

In [21]:
rounds = 15

In [26]:
mw = current_mw
# mw = 1
# url = f'https://api.sofascore.com/api/v1/unique-tournament/240/season/58043/events/round/{mw}'
# response = requests.get(url, headers=headers)

In [105]:
equipos = {
    'Aucas': 'aucas',
    'Barcelona SC': 'barcelona',
    'Cuenca': 'cuenca',
    'Cumbayá FC': 'cumbaya',
    'Delfín': 'delfin',
    'El Nacional': 'nacional',
    'Emelec': 'emelec',
    'Imbabura': 'imbabura',
    'Independiente del Valle': 'independiente',
    'LDU': 'liga',
    'Libertad': 'libertad',
    'Macará': 'macara',
    'Mushuc Runa':'mushuc-runa',
    'Orense': 'orense',
    'Universidad Católica': 'catolica',
    'Técnico': 'tecnico',
    
}

def get_week_matches(mw):

    url = f'https://api.sofascore.com/api/v1/unique-tournament/240/season/58043/events/round/{mw}'
    response = requests.get(url, headers=headers)
    
    matches = []
    for match in response.json()['events']:
        if match['status']['description'] == 'Ended':
            id = match['id']
            local = equipos[match['homeTeam']['shortName']]
            visitante = equipos[match['awayTeam']['shortName']]
            print(id, local, visitante)
            matches.append({'match_id': str(id), 'home': local, 'away': visitante})
            
    return matches

# for i, event in enumerate(matches):
#     response = requests.get(f'https://api.sofascore.com/api/v1/event/{event["match_id"]}/{data}', headers=headers)
    
#     fp = f'{event["local"]}_{event["visitante"]}_{data}.json'
    
#     with open(fp, 'w') as f:
#       json.dump(response.json(), f)
    
#     print(f'Match {i+1}: Saved {fp}')
#     print(f'{time.localtime().tm_hour}h {time.localtime().tm_min}m {time.localtime().tm_sec}s')
    
#     # After last match has been saved, stop waiting
#     if i < len(matches)-1:
#         time.sleep(30)

In [48]:
current_mw

8

In [2]:
# import os
# # Find all matchweek folders
# base_dir = './data/matches'
# matchweek_dirs = os.listdir(base_dir)
# # Select only directories
# matchweek_folders = [x for x in matchweek_dirs if os.path.isdir(os.path.join(base_dir,x))]

# match_files = []
# matchweek_folders

['mw1', 'mw2', 'mw3', 'mw4', 'mw5']

In [5]:
import os
import pandas as pd

base_dir = './data'
lineups = os.path.join(base_dir, 'ligapro_2024_lineups.csv')

df = pd.read_csv(lineups).iloc[:, 1:]

In [60]:
# Unique matches in DB
db_matches = df[['home', 'away', 'matchweek']].drop_duplicates()
m = db_matches.groupby('matchweek').count()
incomplete_matchweeks = m[m['home'] < 8].index.tolist()

# Add week to list if new week occured
for x in range(max(incomplete_matchweeks)+1, current_mw+1, 1):
    incomplete_matchweeks.append(x)

incomplete_matchweeks

[4, 5, 6, 7, 8]

In [134]:
#week 4

matches = get_week_matches(incomplete_matchweeks[3])

12013970 orense mushuc-runa
12013965 aucas cumbaya
12013978 nacional liga
12013969 barcelona cuenca
12013973 independiente libertad
12013959 delfin catolica
12013979 tecnico emelec
12013967 imbabura macara


In [135]:
incomplete_matchweeks[3]

7

In [136]:
mw = incomplete_matchweeks[3]

db_matches_week = db_matches[db_matches['matchweek'] == mw][['home', 'away']]
print(db_matches_week)

sofa_matches_week = pd.DataFrame(matches)
print(sofa_matches_week)
    

Empty DataFrame
Columns: [home, away]
Index: []
   match_id           home         away
0  12013970         orense  mushuc-runa
1  12013965          aucas      cumbaya
2  12013978       nacional         liga
3  12013969      barcelona       cuenca
4  12013973  independiente     libertad
5  12013959         delfin     catolica
6  12013979        tecnico       emelec
7  12013967       imbabura       macara


In [137]:
missing_matches_id = pd.concat([sofa_matches_week, db_matches_week]).drop_duplicates('home', keep=False).to_dict('records')

missing_matches_id

[{'match_id': '12013970', 'home': 'orense', 'away': 'mushuc-runa'},
 {'match_id': '12013965', 'home': 'aucas', 'away': 'cumbaya'},
 {'match_id': '12013978', 'home': 'nacional', 'away': 'liga'},
 {'match_id': '12013969', 'home': 'barcelona', 'away': 'cuenca'},
 {'match_id': '12013973', 'home': 'independiente', 'away': 'libertad'},
 {'match_id': '12013959', 'home': 'delfin', 'away': 'catolica'},
 {'match_id': '12013979', 'home': 'tecnico', 'away': 'emelec'},
 {'match_id': '12013967', 'home': 'imbabura', 'away': 'macara'}]

In [120]:
from pathlib import Path
def pull_sofascore_data(matches_data, data_name, mw_no, headers):
    for i, event in enumerate(matches_data):
        
        fp = f'{event["home"]}_{event["away"]}_{data}.json'
        
        mw_dir = os.path.join('data', 'matches', f'mw{mw_no}')

        fp = os.path.join(mw_dir, fp)

        response = requests.get(f'https://api.sofascore.com/api/v1/event/{event["match_id"]}/{data_name}', headers=headers)

        # Create folder if it doesn't exist
        Path(mw_dir).mkdir(parents=True, exist_ok=True)
        
        with open(fp, 'w') as f:
          json.dump(response.json(), f)
        
        print(f'Match {i+1}: Saved {fp}')
        print(f'{time.localtime().tm_hour}h {time.localtime().tm_min}m {time.localtime().tm_sec}s')
        
        # After last match has been saved, stop waiting
        if i < len(matches_data)-1:
            time.sleep(120)


In [121]:
pull_sofascore_data([{'match_id': '12013980', 'home': 'liga', 'away': 'imbabura'}], 'shotmap', 6, headers) 

Match 1: Saved data\matches\mw6\liga_imbabura_shotmap.json
22h 25m 41s
